# af-confidence：在 Colab／Jupyter 中檢查突變編號

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wangnick1004/af-confidence/blob/main/examples/colab_quickstart.ipynb)

把文獻中蛋白質層級的突變寫法（例如 `E6V`）對應到 AlphaFold DB 模型的正確位置；對不上時說明原因、指出可能的正確位置，並標示證據強度（strong／medium／weak）。

本筆記本示範：

1. 安裝工具
2. 檢查一個突變、檢查一整份清單
3. 用 pandas 整理結果
4. 產生 AlphaFold DB 自訂標註與 ChimeraX 腳本
5. 上傳你自己的清單

> 工具只做編號對應：不判斷突變是否致病，pLDDT 照 AlphaFold DB 的原值顯示。

## 1. 安裝

只需要執行一次。工具本身不需要其他套件（Colab 已內建 pandas）。

In [ ]:
%pip install -q git+https://github.com/wangnick1004/af-confidence

In [ ]:
import af_confidence as afc
print(afc.__version__)

## 2. 檢查一個突變

鐮刀型貧血的突變，教科書寫 **E6V**，但 AlphaFold DB 模型的第 6 個胺基酸是 P。

In [ ]:
report = afc.check("HBB", ["E6V"])
print(afc.to_text(report))

## 3. 檢查一整份清單（建議這樣用）

整份清單一起檢查時，工具能利用多個突變之間的一致性：證據更強，也能抓出「碰巧對上」的突變（標示 `!?`）。

清單可以寫成 `{蛋白質: 突變}`；蛋白質可以是基因名稱或 UniProt 編號，同一個蛋白質會自動合併。

In [ ]:
reports = afc.check_list({
    "HBB": ["E6V"],                        # 鐮刀型貧血，傳統編號
    "SERPINA1": ["E342K", "E264V"],        # α1-抗胰蛋白酶 Z 型與 S 型，成熟蛋白質編號
    "APP": ["A2T", "A21G", "E22G", "E22Q", "D23N"],   # 阿茲海默症文獻的 Aβ 編號
})
print(afc.to_text(reports))

## 4. 用 pandas 整理結果

`to_dataframe` 每個輸入的突變一列，附上最佳建議的位置（`best_position`）、強度與 pLDDT。

In [ ]:
df = afc.to_dataframe(reports)
df[["protein", "input", "status", "best_position", "best_kind", "best_strength", "best_plddt"]]

In [ ]:
# 例如：只看需要注意的突變（對不上，或可能是巧合）
df[df["status"] != "match"][["protein", "input", "status", "best_position", "best_strength"]]

## 5. 在 AlphaFold DB 網頁上標出正確位置

把輸出中的 Track name 和 Residues or ranges 貼進模型頁的「Add your annotation track」對話框。位置已經換算成該模型的編號。

In [ ]:
print(afc.afdb_tracks(reports))

## 6. 產生 ChimeraX 或 PyMOL 腳本

下載 `markers.cxc`（ChimeraX：File → Open）或 `markers.pml`（PyMOL：File → Run Script...）：會下載正確的模型、依 pLDDT 上色，並標出檢查過的位置。注意：腳本會先關閉或刪除軟體中已開啟的內容。

In [ ]:
with open("markers.cxc", "w") as fh:
    fh.write(afc.chimerax_script(reports))
with open("markers.pml", "w") as fh:
    fh.write(afc.pymol_script(reports))

try:
    from google.colab import files     # 只在 Colab 中可用
    files.download("markers.cxc")
    files.download("markers.pml")
except ImportError:
    print("已存成 markers.cxc 和 markers.pml")

## 7. 檢查你自己的清單

清單檔是純文字，每行一個蛋白質加上它的突變，`#` 後面是註解：

```
HBB      E6V            # 某篇論文 Table 1
SERPINA1 E342K, E264V
```

在 Colab 中執行下一格會出現上傳按鈕；在 Jupyter 中請把 `my_file` 改成檔案路徑。

In [ ]:
my_file = None
try:
    from google.colab import files
    uploaded = files.upload()
    my_file = next(iter(uploaded))
except ImportError:
    pass

if my_file:
    my_reports = afc.check_list(my_file)
    print(afc.to_text(my_reports))
    my_df = afc.to_dataframe(my_reports)
    display(my_df)

---

說明文件與原始碼：<https://github.com/wangnick1004/af-confidence>